# PMM 改寫與 GMM 對照實驗
原始檔 `hierarchical_forecasting.ipynb` 完全保留。本檔由原檔複製。以下先提供可執行的修正版與結果，後方保留原教學架構並改為 PMM；原有輸出已清除，避免把歷史分數誤認為本次結果。
**使用專案 `.venv` Python 3.10 核心。** 套件版本見 `experiment_results/requirements-lock.txt`。
**重要修正：** unique_id 必須使用與 S 相同的 ordered categorical 順序；否則 HINT 會將錯誤序列加總。實驗不把小數資料四捨五入，並以相同的訓練、驗證、測試切分比較。


# GMM → PMM 實驗紀錄

產生時間：2026-10-29T13:45:04+08:00。本次總時限：2026-10-29 12:33:57 至 2026-10-29T14:03:57+08:00（台北時間，包含安裝、環境排錯與中斷等待）。

使用者於第一輪結果完成後追加 30 分鐘，延長截止至 14:03:57。第二、第三組仍是第一輪已記錄的設定，沒有新增第四組。第一輪測試結果已公開，因此延伸測試屬於同一測試期的探索性比較，**不是新的獨立確認集**。設定選擇仍只使用 2015 驗證集。第一輪 notebook、報告與選擇紀錄保留在 `experiment_results/round1/`。

目前驗證集選定 `PMM_C`。它的驗證 pooled sCRPS = 0.125204，測試 pooled sCRPS = 0.116767；修正排序的 GMM 分別為 0.120999、0.117362。數值越低越好。各設定的完整數字與限制如下。

原檔 `hierarchical_forecasting.ipynb` 未修改；修改版為 `ＰＰＰ.ipynb`（全形英文字母）。原檔 SHA256：`074872c915b210f84833c8eed11e238856e01ac294d90003d68fc15cf66a6671`。

## 用途、環境與資料

原程式用 NHITS 產生機率預測，再由 HINT BottomUp 加總成一致的階層預測。GMM 是 Gaussian mixture 損失；PMM 是 Poisson mixture 損失，不是另一個獨立預測網路。兩者 API 可接入相同 NHITS/HINT。

使用專案 `.venv` 的 Python 3.10、CPU 單執行緒；完整版本見 `experiment_results/requirements-lock.txt`。NeuralForecast 3.3.0、PyTorch 2.14.1、HierarchicalForecast 1.5.3、UtilsForecast 0.2.17、DatasetsForecast 1.0.2。系統 Python 3.15 沒有套件，因此另建環境。原 notebook 未鎖版本，無法重建當年的精確環境。

TourismLarge：555 條月序列、228 個月、126,540 列，S 為 555 × 304。資料下載來源是 Nixtla 公開資料集；檔案雜湊見 `data_audit.json`。

| 資料用途 | 時段 | 每條序列月份 |
|---|---|---:|
| 梯度訓練 | 1998-01 至 2014-12 | 204 |
| 驗證、早停與設定選擇 | 2015-01 至 2015-12 | 12 |
| 最終測試 | 2016-01 至 2016-12 | 12 |

與原程式一致：`fit(pretest, val_size=12)`；預測測試期時可用已知的 2015 歷史，但不把 2016 目標傳入訓練或選參數。沒有把小數四捨五入、沒有改資料單位。未重跑教學的 AutoARIMA/MinTrace 支線，因本次目標是 GMM 與 PMM。

## 每次修改與實際結果

1. 複製原 notebook、清除複本中的歷史輸出，將 GMM 介面替換為 PMM；原檔保持原樣。
2. 原樣執行 GMM 的資料／建模／訓練／預測 cells 7、8、13–17。第一次在 OpenMP 執行緒初始化失敗（exit 139），第二次解除沙盒後仍 exit 139。改用單執行緒環境後成功；不曾更改原 notebook 的模型程式。
3. PMM 最小試跑完成 2 步訓練及 6,660 列預測。GMM、PMM A 的 200 步診斷也完成，PMM B 的短跑在觀察到 50 步後中斷，以確保先完成正式驗證選擇、最後才測試。
4. 發現序列順序不符：NeuralForecast 依 ID 排序，S 則保留階層順序。修正版使用與 S 一致的 ordered categorical ID，並以斷言檢查。不能單純字母排序 S，因 HINT BottomUp 假設 S 的最後 304 列為底層單位矩陣。
5. 發現逐序列 sCRPS 的零分母問題，保留套件原指標，另記錄 pooled sCRPS；在正式訓練之前決定以後者選設定，沒有根據測試結果調整。
6. 正式預算沿用原設定最多 2,000 步、每 50 步驗證、patience 10。短跑是獨立重訓，不能當成完整訓練。原套件預設的學習率衰減取決於 max_steps，因此短跑與完整訓練也不是單純的續訓關係。
7. 延長時間後，B 因前次沒有權重／optimizer checkpoint，從 seed=1 重新訓練；C 依原計畫執行。每 250 步增加本機恢復 checkpoint，不改優化與最後權重的計分規則。中斷的歷史紀錄保留，完成結果另標 `extension`。

原樣 GMM：狀態 `completed`，實際 2000 步，耗時 151.8 秒。其驗證 pooled sCRPS = 3.548462。另固定同一組原始權重、不重訓，只修正推論順序，驗證 pooled sCRPS = 0.116832（seed=1）。

共同設定：h=12、input_size=24、10 components、robust scaler、month 歷史特徵、seed=1、1,000 samples、99 個訓練／驗證分位數、原預設 NHITS 架構與 batch/window 設定。保留與原程式相同的最後訓練權重策略；原程式未儲存／還原驗證最佳 checkpoint。

| PMM 設定 | 學習率 | mixture weights | 動機 |
|---|---:|---|---|
| A：直接替換 | 0.001 | 固定等權重 | 測試只換損失的效果 |
| B | 0.0003 | 固定等權重 | A 短跑偏高估，檢查較小更新是否穩定 |
| C | 0.0003 | 可學習權重 | 與 B 比較，檢查固定每成分 10% 機率是否過度受限 |

| 版本 | 階段 | 實際步數 | 狀態 | 驗證 pooled sCRPS | 測試 pooled sCRPS |
|---|---|---:|---|---:|---:|
| GMM_fixed | short | 200 | completed | 0.178488 | — |
| PMM_A | short | 200 | completed | 0.395771 | — |
| PMM_B | short | 50 | interrupted | — | — |
| GMM_fixed | formal | 2000 | completed | 0.120999 | 0.117362 |
| PMM_A | formal | 1550 | completed | 0.399246 | 0.409650 |
| PMM_B | formal | 1250 | interrupted | — | — |
| PMM_B | extension | 1400 | completed | 0.375921 | — |
| PMM_C | extension | 2000 | completed | 0.125204 | 0.116767 |


`short` 僅為試跑；`formal`、`extension` 都要求原 2,000 步預算或原早停規則。`completed` 且少於 2,000 步表示依共同規則早停，並非時間不足。`time_limited` 則不能當成已完成完整預算，也不參與完成組的設定選擇。除了第一輪已看過的 A，延伸階段只對驗證集選中的 PMM 計算測試分數，未選中的設定留白。

目前選定：`PMM_C`；選擇依據：`validation Overall pooled_scaled_crps`。最終測試是否完成：`True`。

已完成的 PMM 設定：`['PMM_A', 'PMM_B', 'PMM_C']`；未完成／排除紀錄：`{}`。表格中的 B 中斷列為第一輪歷史，延伸完成的 B 記在另一列。原樣 GMM 最後測試 pooled sCRPS：3.670745，其排序錯誤仍保留，僅作問題重現。

以下為各組**驗證集**的階層偏差／覆蓋率，正偏差代表高估。整體覆蓋率可能掩蓋全國層級問題，因此分開檢查：

| 版本 | 全國平均偏差 | 全國 90% 覆蓋率 | 底層平均偏差 | 底層 90% 覆蓋率 |
|---|---:|---:|---:|---:|
| GMM_fixed | -695.15 | 83.3% | -2.29 | 91.6% |
| PMM_A | 16533.32 | 0.0% | 54.39 | 98.4% |
| PMM_B | 15321.67 | 8.3% | 50.40 | 98.8% |
| PMM_C | -1045.68 | 75.0% | -3.44 | 90.3% |


![驗證分數與全國層級預測](experiment_results/validation_comparison.png)

在相同 learning_rate=0.0003、相同 split、seed 與早停規則下，B（固定權重）驗證分數 0.375921，C（可學習權重）0.125204，降低約 66.7%。B/C 只改 weighted 旗標；該旗標也會增加權重輸出頭，並非相同網路輸出維度。這是本次最直接的改善證據，支持固定等權重的限制是重要因素，但單一訓練種子不能證明唯一原因或跨資料集的一般結論。

固定同一組訓練權重，預先指定預測抽樣 seed=1,2,3,4,5；不重訓、不挑最好 seed，也不改主表結果。 `GMM_fixed_formal` 的測試 pooled sCRPS 平均 0.117242、標準差 0.000150、範圍 0.116999–0.117369。 `PMM_C_extension` 的測試 pooled sCRPS 平均 0.116889、標準差 0.000222、範圍 0.116576–0.117145。 PMM 分數較低的抽樣 seed 有 5/5 個。這只量測預測抽樣波動，不是不同訓練種子的穩定性，也不是統計顯著性檢定。

![測試期全國層級區間](experiment_results/test_country_intervals.png)

## 指標與數值檢查

主要比較值採 `pooled_scaled_crps = 2 × Σ pinball(y, q) / (98 × Σ |y|)`，加總範圍為該階層所有序列與 12 個月，98 個 q 與原評估的 level=2,4,…,98 對應（不含中位數）。這是以總絕對目標作分母的尺度化 CRPS 近似；不同於目前套件先除每條序列各自分母再平均。原 notebook 的歷史 Overall=0.122793 只是舊輸出，不能拿來當本次基準，也不能假定套件版本改變前後指標實作完全相同。

已額外確認：原 notebook 儲存的八個階層分數平均為 0.12279275，四捨五入後正好等於原 Overall。本次非負資料每層的目標總量相同，因此 pooled Overall 也等於八層 pooled 分數的平均。這支持指標彙總方式的對照，但無法識別當年的套件版本。另以簡單解析例子驗證目前 NeuralForecast 內部 sCRPS：理論值 0.1，套件回傳約 9.9（99 個分位數）。此共同倍率不改同一分位數設定下的排序或零 min_delta 早停，不是 PMM 相對較差的解釋；本次外部指標沒有這個倍率。

`*_metrics.csv` 保留原套件 scaled_crps、MAE、RMSE；`*_pooled_metrics.csv` 存穩定的共同指標。`*_diagnostics.csv` 提供各層 90% 區間覆蓋率、寬度、偏差與負平均預測數；`*_predictions.parquet` 保留實際預測。比較前檢查列數、ID／日期、finite 值、分位數單調性與階層平均值一致性。`verification.json` 另以套件在合併 ID 後的 scaled_crps 交叉驗證手算 pooled 公式。

## 已確認的事實

- 實際資料沒有負值，但 112973 筆（89.28%）不是整數。原程式在切分後才歸零負值，對已切出的訓練資料不起作用；此資料沒有負值，所以本次不是造成差異的原因。
- PMM 程式可以接受非負小數；0.25 的 likelihood 與梯度試算均為有限值。套件把 Poisson support 放寬成 nonnegative，並以 gamma 函數延伸計算。因此「能執行」不代表這是符合資料生成過程的離散機率模型。
- 原程式的 ID 順序與 S 不符；固定同一組已訓練權重，只改推論順序即可大幅改善驗證分數，確認排序／階層對應有實質影響。
- 驗證集有全零序列，套件逐序列 scaled_crps 的 epsilon 分母會使短跑分數達 10^13 等級。原指標仍完整保留；兩版都用同一個 pooled 定義作主要比較。
- 修正後的預測平均值通過階層加總檢查；沒有把分位數誤當成可直接相加的量。
- 官方 PMM 支援 n_components、quantiles、num_samples、weighted 等參數；本次沒有修改套件原始碼。[Nixtla PMM 文件](https://nixtlaverse.nixtla.io/neuralforecast/losses.pytorch.html#poisson-mixture-mesh-pmm)
- 實測 C 的可學習權重與 B 的固定權重有明顯效果差異，見上方控制條件與數值。A/B 的偏高估經 BottomUp 累加至全國，C 已大幅縮小此偏差。

## 可能原因與尚未確認的部分

- 旅遊資料多為實數估計值，Poisson 的離散計數假設可能不匹配；不能單憑小數比例證明它是誤差主因。
- 每個 Poisson 成分的平均數與變異數綁定；混合後仍可以過度離散，不能簡化成「PMM 的變異數必等於平均數」。相較 GMM 的個別可學習標準差，PMM 可能需要不同的成分位置／權重才能表達尾端、零值及季節變動。
- 訓練期逐序列樣本變異數／平均數中位數約 44.66。這包含季節性與時間變化，**不是條件過度離散的證明**。
- PMM A 短跑有偏高估現象；rate 下界與混合權重診斷見 `*_rate_diagnostics.json`。softplus 下界可能影響梯度，但本次僅檢查預測參數，不能斷言發生梯度消失或成分崩潰。
- A 完整訓練的 rate 成分觸及下界比例約 0.88%，且沒有任何驗證預測點的全部成分同時在下界，故目前沒有支持「整個模型卡在零 rate」的證據。後驗責任集中在少數成分是混合模型的常見現象，不能單憑此就判定成分失效。
- HINT 用 bootstrap 重排樣本再 BottomUp 加總，無法據此證明所有跨序列依賴都建模正確；此限制對本次兩版相同。
- 未提供先前效果差的 PMM notebook、當時版本或紀錄，因此無法斷定先前是哪一個原因。只使用單一**訓練** seed；額外 5 個 seed 僅重抽固定模型的預測樣本，不能代替多次訓練或統計顯著性結論。

## 重新執行

在編輯器選擇 `.venv/bin/python` 作為 notebook 核心。上方實驗定義是實際執行的 `experiment_code.py` 原碼副本；下方留存教學片段為參考，不會在 Run All 時啟動第二套模型或 AutoARIMA。預設只讀取已記錄結果，避免重跑覆蓋。若要重訓，先另備份 `experiment_results`，再用 `run_experiment(steps=2000, budget_seconds=3600, phase='formal', evaluate_test=True)`；先驗證用 `evaluate_test=False`。

不必重新訓練即可使用已保存權重：`predict_saved_model(run_id, history, S)`；函式會重建相同設定、核對階層順序，並預測下一個 12 個月。選定 run_id 記錄於 `selection.json` 的 `run_ids`。Notebook 內附範例。三組 PMM 的訓練配置已用完本次搜尋上限，沒有另外試第四組。

原樣重跑器：`original_gmm_attempt.py`；驗證順序隔離實驗：`isolate_ordering.py`；參數診斷：`inspect_rates.py`；檢查：`verify_results.py`。原樣重跑器需設定與本次相同的 OMP_NUM_THREADS=1、OPENBLAS_NUM_THREADS=1、MKL_NUM_THREADS=1、KMP_DUPLICATE_LIB_OK=TRUE。


In [1]:
"""Source embedded verbatim in ＰＰＰ.ipynb; all outputs are local."""
import os
from pathlib import Path
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
os.environ.setdefault('MPLCONFIGDIR', str(Path('.mpl-cache').resolve()))

import json
import time
import hashlib
import traceback
import numpy as np
import pandas as pd
import torch
from pytorch_lightning import Callback
from datasetsforecast.hierarchical import HierarchicalData
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, HINT
from neuralforecast.losses.pytorch import GMM, PMM, sCRPS
from neuralforecast.utils import augment_calendar_df
from neuralforecast.tsdataset import TimeSeriesDataset
from hierarchicalforecast.evaluation import evaluate
from utilsforecast.losses import scaled_crps, mae, rmse

torch.set_num_threads(1)
OUT = Path('experiment_results')
OUT.mkdir(exist_ok=True)
HORIZON = 12
LEVELS = np.arange(2, 100, 2).tolist()
QUANTILES = np.arange(1, 100) / 100
SEED = 1

def write_json(path, value):
    path = Path(path)
    temporary = path.with_name(path.name + f'.tmp-{os.getpid()}')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                   default=lambda x: float(x) if np.isscalar(x) else str(x)))
    temporary.replace(path)

def load_data():
    y, s, tags = HierarchicalData.load('./data', 'TourismLarge')
    y['ds'] = pd.to_datetime(y['ds'])
    y, _ = augment_calendar_df(df=y, freq='M')
    if 'unique_id' in s.columns:
        s = s.set_index('unique_id')
    # HINT BottomUp requires S's original order, with bottom identities last.
    # Merely sorting S alphabetically would break BottomUp's bottom-row assumption.
    raw_ids = TimeSeriesDataset.from_df(y)[1]
    raw_match = list(raw_ids) == list(s.index)
    dtype = pd.CategoricalDtype(categories=s.index.tolist(), ordered=True)
    y['unique_id'] = y['unique_id'].astype(dtype)
    y = y.sort_values(['unique_id', 'ds']).reset_index(drop=True)
    assert list(TimeSeriesDataset.from_df(y)[1]) == list(s.index)
    assert np.allclose(s.iloc[-s.shape[1]:].to_numpy(), np.eye(s.shape[1]))
    assert not y.duplicated(['unique_id', 'ds']).any()
    assert np.isfinite(y['y']).all() and (y['y'] >= 0).all()
    # Preserve original real-valued observations: no rounding or unit conversion.
    test = y.groupby('unique_id', observed=True).tail(HORIZON).copy()
    pretest = y.drop(test.index).copy()
    valid = pretest.groupby('unique_id', observed=True).tail(HORIZON).copy()
    train = pretest.drop(valid.index).copy()
    for frame in [y, pretest, train, valid, test]:
        assert frame.groupby('unique_id', observed=True).size().nunique() == 1
    stats = train.groupby('unique_id', observed=True)['y'].agg(['mean', 'var'])
    audit = {
        'series': int(y.unique_id.nunique()), 'rows': len(y),
        'S_shape': list(s.shape), 'negative_count': int((y.y < 0).sum()),
        'fractional_count': int((np.abs(y.y - np.round(y.y)) > 1e-6).sum()),
        'fractional_fraction': float((np.abs(y.y - np.round(y.y)) > 1e-6).mean()),
        'zero_fraction_train': float((train.y == 0).mean()),
        'raw_NeuralForecast_order_matches_S': raw_match,
        'fixed_order_matches_S': True,
        'train_variance_over_mean_median': float((stats['var'] / stats['mean'].replace(0, np.nan)).median()),
        'variance_ratio_is_unconditional_not_proof_of_conditional_overdispersion': True,
        'splits': {name: {'start':str(df.ds.min().date()),'end':str(df.ds.max().date()),
                           'rows':len(df), 'months_per_series':len(df)//len(s)}
                   for name, df in [('train',train),('validation',valid),('test',test)]},
        'data_sha256': hashlib.sha256(Path('data/hierarchical/TourismLarge/data.csv').read_bytes()).hexdigest(),
    }
    # Check observed hierarchy using training data only.
    pivot = train.pivot(index='unique_id', columns='ds', values='y').reindex(s.index)
    audit['observed_hierarchy_max_abs_error_train'] = float(np.abs(pivot.to_numpy()-s.to_numpy()@pivot.reindex(s.columns).to_numpy()).max())
    write_json(OUT/'data_audit.json', audit)
    print('DATA_AUDIT', json.dumps(audit, ensure_ascii=False), flush=True)
    return y, s, tags, train, valid, pretest, test

class Progress(Callback):
    def __init__(self, name, deadline):
        self.name, self.deadline = name, deadline
        self.started = time.time()
        self.records = []
        self.timed_out = False

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        if trainer.global_step % 50 == 0:
            loss = outputs['loss'] if isinstance(outputs, dict) else outputs
            rec = {'step':trainer.global_step,'loss':float(loss.detach().cpu()),
                   'seconds':time.time()-self.started}
            self.records.append(rec)
            write_json(OUT/f'{self.name}_progress.json', self.records)
            print(self.name, rec, flush=True)
        if time.time() >= self.deadline:
            self.timed_out = True
            trainer.should_stop = True

    def on_validation_end(self, trainer, pl_module):
        # Local recovery checkpoint only; do not change optimization or restore
        # another epoch when comparing against the original final-weight policy.
        if not trainer.sanity_checking and trainer.global_step and trainer.global_step % 250 == 0:
            path = OUT / f'{self.name}_latest.ckpt'
            temporary = path.with_suffix('.ckpt.tmp')
            trainer.save_checkpoint(temporary)
            temporary.replace(path)
            write_json(OUT/f'{self.name}_checkpoint.json',
                       {'step':trainer.global_step,'file':str(path),'saved_epoch':time.time()})

def score(pred, truth, tags, history, name, phase):
    pred = pred.rename(columns=lambda c: c.replace('.0', ''))
    joined = truth[['unique_id','ds','y']].merge(pred, on=['unique_id','ds'], validate='one_to_one')
    assert len(joined) == len(truth)
    columns = [c for c in pred.columns if c not in ['unique_id','ds']]
    assert np.isfinite(joined[columns].to_numpy()).all()
    result = evaluate(df=joined, tags=tags, metrics=[scaled_crps, mae, rmse],
                      models=['NHITS'], level=LEVELS,
                      train_df=history[['unique_id','ds','y']])
    result.to_csv(OUT/f'{name}_{phase}_metrics.csv', index=False)
    diagnostics = []
    for level, ids in {**tags, 'Overall':np.concatenate(list(tags.values()))}.items():
        part = joined[joined.unique_id.isin(ids)]
        diagnostics.append({'level':level,
            'coverage90': float(((part.y>=part['NHITS-lo-90']) & (part.y<=part['NHITS-hi-90'])).mean()),
            'width90': float((part['NHITS-hi-90']-part['NHITS-lo-90']).mean()),
            'bias': float((part['NHITS']-part.y).mean()),
            'negative_mean_count':int((part['NHITS']<0).sum())})
    pd.DataFrame(diagnostics).to_csv(OUT/f'{name}_{phase}_diagnostics.csv',index=False)
    qs = [f'NHITS-lo-{lv}' for lv in reversed(LEVELS)] + ['NHITS-median'] + [f'NHITS-hi-{lv}' for lv in LEVELS]
    assert (np.diff(joined[qs].to_numpy(),axis=1)>=-1e-6).all()
    # Current utilsforecast averages per-series ratios. An all-zero truth series
    # divides by machine epsilon and can dominate the result (~1e13). Retain
    # that exact library metric above, and also compute the pooled definition
    # written in the original notebook. Use the same 98 interval quantiles.
    pooled_rows=[]
    eval_q=np.concatenate([np.arange(1,50),np.arange(51,100)])/100
    eval_cols=[c for c in qs if c!='NHITS-median']
    for level, ids in {**tags,'Overall':np.concatenate(list(tags.values()))}.items():
        part=joined[joined.unique_id.isin(ids)]
        err=part['y'].to_numpy()[:,None]-part[eval_cols].to_numpy()
        pinball=np.maximum(eval_q*err,(eval_q-1)*err)
        denom=float(part.y.abs().sum())
        pooled=float(2*pinball.sum()/(len(eval_q)*denom)) if denom>0 else None
        pooled_rows.append({'level':level,'metric':'pooled_scaled_crps','NHITS':pooled,
                            'zero_total_series':int((part.groupby('unique_id',observed=True).y.sum()==0).sum())})
    pd.DataFrame(pooled_rows).to_csv(OUT/f'{name}_{phase}_pooled_metrics.csv',index=False)
    return next(row['NHITS'] for row in pooled_rows if row['level']=='Overall')

def fit_run(config, steps, data, deadline, phase='short'):
    y, s, tags, train, valid, pretest, test = data
    name = config['name']
    run_id = f'{name}_{phase}'
    cb = Progress(run_id, deadline)
    loss_cls = GMM if config['loss']=='GMM' else PMM
    loss = loss_cls(n_components=config.get('components',10), quantiles=QUANTILES,
                    weighted=config.get('weighted',False), num_samples=1000)
    base = NHITS(h=12,input_size=24,loss=loss,hist_exog_list=['month'],
                 max_steps=steps,early_stop_patience_steps=10,val_check_steps=min(50,steps),
                 scaler_type='robust',learning_rate=config['lr'],
                 valid_loss=sCRPS(quantiles=QUANTILES),random_seed=SEED,
                 accelerator='cpu',devices=1,logger=False,enable_progress_bar=False,
                 enable_model_summary=False,callbacks=[cb])
    model = HINT(h=12,S=s.to_numpy(),model=base,reconciliation='BottomUp',alias='NHITS')
    nf = NeuralForecast(models=[model],freq='MS')
    record = {**config,'phase':phase,'max_steps':steps,'seed':SEED,
              'num_samples':1000,'input_size':24,'scaler':'robust',
              'val_check_steps':min(50,steps),'early_stop_patience_steps':10,
              'started_epoch':time.time(),'test_evaluated':False,
              'ordering_fixed':True,'checkpoint_policy':'final weights, matching original notebook'}
    write_json(OUT/f'{run_id}.json',record)
    try:
        nf.fit(df=pretest,val_size=12)
        trained=nf.models[0].model
        assert list(nf.uids)==list(s.index)
        record['actual_steps']=len(trained.train_trajectories)
        record['train_trajectories']=trained.train_trajectories
        record['valid_trajectories']=trained.valid_trajectories
        record['timed_out']=cb.timed_out
        record['status']='completed' if not cb.timed_out else 'time_limited'
        record['stop_reason']='time_limit' if cb.timed_out else (
            'max_steps' if record['actual_steps']>=steps else 'early_stopping')
        np.random.seed(SEED)
        pred=nf.predict(df=train,random_seed=SEED)
        pred.to_parquet(OUT/f'{run_id}_validation_predictions.parquet')
        record['validation_scrps']=score(pred,valid,tags,train,run_id,'validation')
        torch.save(trained.state_dict(),OUT/f'{run_id}_weights.pt')
        # Reconciled point means must match the summing matrix for the fixed order.
        pivot=pred.pivot(index='unique_id',columns='ds',values='NHITS').reindex(s.index)
        record['mean_coherence_max_abs_error']=float(np.abs(pivot.to_numpy()-s.to_numpy()@pivot.reindex(s.columns).to_numpy()).max())
        print('FINISHED',run_id,record['actual_steps'],record['validation_scrps'],flush=True)
    except Exception as exc:
        record.update(status='failed',error=repr(exc))
        traceback.print_exc()
        nf=None
    record['elapsed_seconds']=time.time()-record['started_epoch']
    write_json(OUT/f'{run_id}.json',record)
    return nf,record

def predict_saved_model(run_id, history, S, sampling_seed=None):
    """Reload a local fitted model and forecast 12 months without retraining.

    `history` contains unique_id/ds/y/month. S is indexed by unique_id, in its
    original hierarchy order. No evaluation targets are needed or accessed.
    """
    record=json.loads((OUT/f'{run_id}.json').read_text())
    if record.get('status')!='completed':
        raise ValueError(f'{run_id} is not a completed run')
    history=history.copy()
    if set(history.unique_id.astype(str))!=set(S.index.astype(str)):
        raise ValueError('History must contain every series in the hierarchy')
    history['unique_id']=history['unique_id'].astype(
        pd.CategoricalDtype(categories=S.index.tolist(),ordered=True))
    history=history.sort_values(['unique_id','ds'])
    loss_cls=GMM if record['loss']=='GMM' else PMM
    base=NHITS(h=12,input_size=24,
        loss=loss_cls(n_components=record.get('components',10),quantiles=QUANTILES,
                      weighted=record.get('weighted',False),num_samples=1000),
        hist_exog_list=['month'],max_steps=record['max_steps'],
        scaler_type='robust',learning_rate=record['lr'],valid_loss=sCRPS(quantiles=QUANTILES),
        random_seed=record['seed'],accelerator='cpu',devices=1,logger=False,
        enable_progress_bar=False,enable_model_summary=False)
    base.load_state_dict(torch.load(OUT/f'{run_id}_weights.pt',weights_only=True))
    hint=HINT(h=12,S=S.to_numpy(),model=base,reconciliation='BottomUp',alias='NHITS')
    dataset,uids,last_dates,*_=TimeSeriesDataset.from_df(history)
    assert list(uids)==list(S.index)
    inference_seed=record['seed'] if sampling_seed is None else sampling_seed
    np.random.seed(inference_seed)
    values=hint.predict(dataset,random_seed=inference_seed)
    dates=np.concatenate([pd.date_range(pd.Timestamp(date)+pd.offsets.MonthBegin(1),
                          periods=12,freq='MS').to_numpy() for date in last_dates])
    pred=pd.DataFrame({'unique_id':np.repeat(np.asarray(uids),12),'ds':dates})
    pred['unique_id']=pred['unique_id'].astype(history['unique_id'].dtype)
    return pd.concat([pred,pd.DataFrame(values,columns=['NHITS'+c for c in base.loss.output_names])],axis=1)

def run_experiment(steps=200, budget_seconds=600, phase='short', evaluate_test=False):
    data=load_data()
    # At most 3 PMM settings. No rounding, no test-dependent decisions.
    configs=[{'name':'GMM_fixed','loss':'GMM','lr':1e-3},
             {'name':'PMM_A','loss':'PMM','lr':1e-3},
             {'name':'PMM_B','loss':'PMM','lr':3e-4},
             {'name':'PMM_C','loss':'PMM','lr':3e-4,'weighted':True}]
    deadline=time.time()+budget_seconds
    runs={}
    for config in configs:
        if time.time()>deadline-45:
            break
        nf,rec=fit_run(config,steps,data,deadline-30,phase)
        if nf is not None:
            runs[config['name']]=(nf,rec)
    pmm=[(k,r) for k,(nf,r) in runs.items() if r['loss']=='PMM' and 'validation_scrps' in r]
    if not pmm or 'GMM_fixed' not in runs:
        write_json(OUT/'selection.json',{'status':'incomplete','phase':phase,'test_evaluated':False})
        return runs
    # Compare only equal budgets without timeout. Never compare an incomplete run
    # as if it had received the same training budget.
    pmm=[(k,r) for k,r in pmm if not r.get('timed_out')]
    if not pmm:
        return runs
    selected=min(pmm,key=lambda item:item[1]['validation_scrps'])[0]
    selection={'selected':selected,'phase':phase,'max_steps':steps,
               'selection_metric':'validation Overall pooled_scaled_crps after HINT BottomUp; native scaled_crps also retained',
               'validation_scores':{k:r['validation_scrps'] for k,r in pmm},
               'selected_at_epoch':time.time(),'test_evaluated':False}
    write_json(OUT/'selection.json',selection)
    if not evaluate_test:
        print('VALIDATION_ONLY', selection, flush=True)
        return runs
    # Only now use test targets, once for the fixed baseline and chosen PMM.
    y,s,tags,train,valid,pretest,test=data
    for name in ['GMM_fixed',selected]:
        nf,record=runs[name]
        np.random.seed(SEED)
        pred=nf.predict(df=pretest,random_seed=SEED)
        run_id=f'{name}_{phase}'
        pred.to_parquet(OUT/f'{run_id}_test_predictions.parquet')
        record['test_scrps']=score(pred,test,tags,pretest,run_id,'test')
        record['test_evaluated']=True
        write_json(OUT/f'{run_id}.json',record)
        print('FINAL_TEST',run_id,record['test_scrps'],flush=True)
    selection['test_evaluated']=True
    write_json(OUT/'selection.json',selection)
    return runs


/Users/cys/Desktop/meeting/pmm/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-29 13:45:51,634	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-10-29 13:45:52,026	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [2]:
# 預設讀取已完成的本次實驗，不會重新訓練。
RUN_EXPERIMENT = False
if RUN_EXPERIMENT:
    runs = run_experiment(steps=2000, budget_seconds=3600, phase="formal", evaluate_test=True)
else:
    display(pd.read_csv(OUT / "comparison.csv"))
    print("驗證集選擇：", json.loads((OUT / "selection.json").read_text()) if (OUT / "selection.json").exists() else "尚未完成")


,run,phase,status,actual_steps,validation_pooled_scrps,test_pooled_scrps,learning_rate,weighted,timed_out
0,GMM_fixed,short,completed,200,0.178488,NaN,0.0010,False,False
1,PMM_A,short,completed,200,0.395771,NaN,0.0010,False,False
2,PMM_B,short,interrupted,50,NaN,NaN,0.0003,False,False
3,GMM_fixed,formal,completed,2000,0.120999,0.117362,0.0010,False,False
4,PMM_A,formal,completed,1550,0.399246,0.409650,0.0010,False,False
5,PMM_B,formal,interrupted,1250,NaN,NaN,0.0003,False,False
6,PMM_B,extension,completed,1400,0.375921,NaN,0.0003,False,False
7,PMM_C,extension,completed,2000,0.125204,0.116767,0.0003,True,False


驗證集選擇： {'selected': 'PMM_C', 'phase': 'extension', 'max_steps': 2000, 'selection_metric': 'validation Overall pooled_scaled_crps', 'validation_scores': {'PMM_A': 0.3992463528885444, 'PMM_B': 0.37592133372097997, 'PMM_C': 0.1252036383279902}, 'run_ids': {'PMM_A': 'PMM_A_formal', 'PMM_B': 'PMM_B_extension', 'PMM_C': 'PMM_C_extension'}, 'excluded': {}, 'selected_at_epoch': 1793252498.640075, 'test_evaluated': True, 'previous_test_results_already_seen': True, 'limitation': 'Extension of predeclared validation search; test period is reused and is not a fresh independent confirmation.', 'test_scrps': 0.11676661386000264, 'finished_epoch': 1793252506.70316}


In [3]:
# 讀取驗證集選出的設定；預設不重新訓練，也不重新計分測試集。
selection = json.loads((OUT / "selection.json").read_text())
selected_run_id = selection.get("run_ids", {}).get(selection["selected"], selection["selected"] + "_formal")
selected_config = json.loads((OUT / f"{selected_run_id}.json").read_text())
print("已選定模型：", selected_run_id)
print({key: selected_config.get(key) for key in ["loss", "lr", "weighted", "actual_steps", "validation_scrps", "test_scrps"]})
RUN_SAVED_PREDICTION = False
if RUN_SAVED_PREDICTION:
    Y, S, tags, train, validation, pretest, test = load_data()
    saved_forecast = predict_saved_model(selected_run_id, pretest, S)
    display(saved_forecast.head())


已選定模型： PMM_C_extension
{'loss': 'PMM', 'lr': 0.0003, 'weighted': True, 'actual_steps': 2000, 'validation_scrps': 0.1252036383279902, 'test_scrps': 0.11676661386000264}


## 原教學內容的 PMM 改寫版本（參考）
下方教學程式為留存的改寫範例，不用於本次計分；執行實驗請使用上方 `run_experiment`。


# Hierarchical Forecast | NeuralForecast

> A minimal example of using Hierarchical Forecast with NeuralForecast

This notebook offers a step by step guide to create a hierarchical forecasting pipeline.

In the pipeline we will use `NeuralForecast` and `HINT` class, to create fit, predict and reconcile forecasts.

We will use the TourismL dataset that summarizes large Australian national visitor survey.

Outline<br>
1. Installing packages<br>
2. Load hierarchical dataset<br>
3. Fit and Predict HINT<br>
4. Benchmark methods<br>
5. Forecast Evaluation

You can run these experiments using GPU with Google Colab.

<a href="https://colab.research.google.com/github/Nixtla/neuralforecast/blob/main/nbs/docs/tutorials/hierarchical_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Installing packages

```python
%%capture
!pip install datasetsforecast hierarchicalforecast neuralforecast statsforecast
```


## 2. Load hierarchical dataset

This detailed Australian Tourism Dataset comes from the National Visitor Survey, managed by the Tourism Research Australia, it is composed of 555 monthly series from 1998 to 2016, it is organized geographically, and purpose of travel. The natural geographical hierarchy comprises seven states, divided further in 27 zones and 76 regions. The purpose of travel categories are holiday, visiting friends and relatives (VFR), business and other. The MinT (Wickramasuriya et al., 2019), among other hierarchical forecasting studies has used the dataset it in the past. The dataset can be accessed in the [MinT reconciliation webpage](https://robjhyndman.com/publications/mint/), although other sources are available.

| Geographical Division | Number of series per division | Number of series per purpose | Total |
|          ---          |               ---             |              ---             |  ---  |
|  Australia            |              1                |               4              |   5   |
|  States               |              7                |              28              |  35   |
|  Zones                |             27                |              108             |  135  |
|  Regions              |             76                |              304             |  380  |
|  Total                |            111                |              444             |  555  |


```python
import pandas as pd

from datasetsforecast.hierarchical import HierarchicalData
from hierarchicalforecast.utils import aggregate, HierarchicalPlot
from neuralforecast.utils import augment_calendar_df
from utilsforecast.plotting import plot_series
```


```python
# Load hierarchical dataset
Y_df, S_df, tags = HierarchicalData.load('./data', 'TourismLarge')
Y_df['ds'] = pd.to_datetime(Y_df['ds'])
Y_df, _ = augment_calendar_df(df=Y_df, freq='M')
S_df = S_df.reset_index(names="unique_id")
```


Mathematically a hierarchical multivariate time series can be denoted by the vector $\mathbf{y}_{[a,b],t}$ defined by the following aggregation constraint:

$$
\mathbf{y}_{[a,b],t}  = \mathbf{S}_{[a,b][b]} \mathbf{y}_{[b],t} \quad \Leftrightarrow \quad
\begin{bmatrix}\mathbf{y}_{[a],t}
\\ %\hline
\mathbf{y}_{[b],t}\end{bmatrix}
= \begin{bmatrix}
\mathbf{A}_{[a][b]}\\ %\hline
\mathbf{I}_{[b][b]}
\end{bmatrix}
\mathbf{y}_{[b],t}
$$

where $\mathbf{y}_{[a],t}$ are the aggregate series, $\mathbf{y}_{[b],t}$ are the bottom level series and $\mathbf{S}_{[a,b][b]}$ are the hierarchical aggregation constraints.

```python
# Here we plot the hierarchical constraints matrix
hplot = HierarchicalPlot(S=S_df, tags=tags)
hplot.plot_summing_matrix()
```


```python
plot_series(forecasts_df=Y_df[["unique_id", "ds", "y"]], ids=['TotalAll'])
```


## 3. Fit and Predict HINT

The Hierarchical Forecast Network (HINT) combines into an easy to use model three components:<br>
1. SoTA neural forecast model.<br>
2. An efficient and flexible multivariate probability distribution.<br>
3. Builtin reconciliation capabilities.<br>

```python
import logging

import numpy as np

from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, HINT
from neuralforecast.losses.pytorch import PMM, sCRPS
```


```python
# Train test splits
horizon = 12
Y_test_df  = Y_df.groupby('unique_id', observed=True).tail(horizon)
Y_train_df = Y_df.drop(Y_test_df.index)
```


```python
# Horizon and quantiles
level = np.arange(0, 100, 2)
qs = [[50-lv/2, 50+lv/2] if lv!=0 else [50] for lv in level]
quantiles = np.sort(np.concatenate(qs)/100)

# HINT := BaseNetwork + Distribution + Reconciliation
nhits = NHITS(h=horizon,
              input_size=24,
              loss=PMM(n_components=10, quantiles=quantiles),
              hist_exog_list=['month'],
              max_steps=2000,
              early_stop_patience_steps=10,
              val_check_steps=50,
              scaler_type='robust',
              learning_rate=1e-3,
              valid_loss=sCRPS(quantiles=quantiles))

model = HINT(h=horizon, S=S_df.drop(columns='unique_id').values,
             model=nhits,  reconciliation='BottomUp')
```


```python
logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)
```


```python
%%capture
Y_df['y'] = Y_df['y'] * (Y_df['y'] > 0)
nf = NeuralForecast(models=[model], freq='MS')
nf.fit(df=Y_train_df, val_size=12)
Y_hat_df = nf.predict()

Y_hat_df = Y_hat_df.rename(columns=lambda x: x.replace('.0', ''))
```


```python
plot_series(
    Y_df,
    Y_hat_df.drop(columns='NHITS-median'),
    ids=['TotalAll'],
    level=[90],
    max_insample_length=12*5,
)
```


## 4. Benchmark methods

We compare against AutoARIMA, a well-established traditional forecasting method from the [StatsForecast](../../../statsforecast/index.html) package, for which we reconcile the forecasts using [HierarchicalForecast](../../../hierarchicalforecast/index.html).

```python
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from hierarchicalforecast.methods import BottomUp, MinTrace
from hierarchicalforecast.core import HierarchicalReconciliation
```


We define the model, and create the forecasts.

```python
sf = StatsForecast(models=[AutoARIMA()],
                     freq='MS', n_jobs=-1)
Y_hat_df_arima = sf.forecast(df=Y_train_df,
                             h=12,
                             fitted=True,
                             X_df=Y_test_df.drop(columns="y"),
                             level = np.arange(2, 100, 2))
Y_fitted_df_arima = sf.forecast_fitted_values()
```


Next, we reconcile the forecasts using `BottomUp` and `MinTrace` reconciliation techniques:

```python
reconcilers = [
    BottomUp(),
    MinTrace(method='mint_shrink'),
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(Y_hat_df=Y_hat_df_arima,
                          Y_df=Y_fitted_df_arima,
                          S=S_df,
                          tags=tags,
                          level = np.arange(2, 100, 2),
                          intervals_method="bootstrap")
```


## 5. Forecast Evaluation

To evaluate the coherent probabilistic predictions we use the scaled Continuous Ranked Probability Score (sCRPS), defined as follows:

$$
\mathrm{CRPS}(\hat{F}_{[a,b],\tau},\mathbf{y}_{[a,b],\tau}) =
    \frac{2}{N_{a}+N_{b}} \sum_{i} \int^{1}_{0} \mathrm{QL}(\hat{F}_{i,\tau}, y_{i,\tau})_{q} dq
$$

$$
\mathrm{sCRPS}(\hat{F}_{[a,b\,],\tau},\mathbf{y}_{[a,b\,],\tau}) =
    \frac{\mathrm{CRPS}(\hat{F}_{[a,b\,],\tau},\mathbf{y}_{[a,b\,],\tau})}{\sum_{i} | y_{i,\tau} |}
$$

As you can see the HINT model (using NHITS as base model) efficiently achieves state of the art accuracy under minimal tuning.

```python
from utilsforecast.losses import scaled_crps
from hierarchicalforecast.evaluation import evaluate
```


```python
df_metrics = Y_hat_df.merge(Y_test_df.drop(columns="month"), on=['unique_id', 'ds'])
df_metrics = df_metrics.merge(Y_rec_df, on=['unique_id', 'ds'])

metrics = evaluate(df = df_metrics,
                    tags = tags,
                    metrics = [scaled_crps],
                    models= ["NHITS", "AutoARIMA"],
                    level = np.arange(2, 100, 2),
                    train_df = Y_train_df.drop(columns="month"),
                    )

metrics
```


## References

- [Kin G. Olivares, David Luo, Cristian Challu, Stefania La Vattiata, Max Mergenthaler, Artur Dubrawski (2023). "HINT: Hierarchical Mixture Networks For Coherent Probabilistic Forecasting". International Conference on Machine Learning (ICML). Workshop on Structured Probabilistic Inference & Generative Modeling. Available at https://arxiv.org/abs/2305.07089.](https://arxiv.org/abs/2305.07089)<br />
- [Kin G. Olivares, O. Nganba Meetei, Ruijun Ma, Rohan Reddy, Mengfei Cao, Lee Dicker (2023)."Probabilistic Hierarchical Forecasting with Deep Poisson Mixtures". International Journal Forecasting, accepted paper. URL https://arxiv.org/pdf/2110.13179.pdf.](https://arxiv.org/pdf/2110.13179.pdf)<br />
- [Kin G. Olivares, Federico Garza, David Luo, Cristian Challu, Max Mergenthaler, Souhaib Ben Taieb, Shanika Wickramasuriya, and Artur Dubrawski (2023). "HierarchicalForecast: A reference framework for hierarchical forecasting". Journal of Machine Learning Research, submitted. URL https://arxiv.org/abs/2207.03517](https://arxiv.org/abs/2207.03517)